# ERCOT Houston Power Market Analysis, 2021–2026

This notebook is a guided exploration of the SQLite database produced by `src/data_pipeline.py`. The full, reproducible figures are generated by `src/analysis.py`.

**Research question:** How have ERCOT demand, generation mix, and Houston day-ahead prices evolved, and under which supply-demand conditions do high prices occur?

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATABASE = ROOT / 'data' / 'ercot_market.sqlite'

with sqlite3.connect(DATABASE) as connection:
    market = pd.read_sql_query(
        'SELECT * FROM market_features ORDER BY timestamp', connection, parse_dates=['timestamp']
    )

market.shape, market[['timestamp', 'dam_price_usd_mwh', 'system_load_mw', 'wind_mw', 'solar_mw']].head()

## 1. Annual market evolution

The following query keeps the SQL work visible: prices, load, and generation were joined in the `market_hourly` view, then `market_features` added annual relative regimes and a P95 spike flag.

In [ ]:
with sqlite3.connect(DATABASE) as connection:
    annual = pd.read_sql_query('''
        SELECT year, COUNT(*) AS observations,
               ROUND(AVG(system_load_mw), 1) AS average_load_mw,
               ROUND(MAX(system_load_mw), 1) AS peak_load_mw,
               ROUND(AVG(wind_mw), 1) AS average_wind_mw,
               ROUND(AVG(solar_mw), 1) AS average_solar_mw,
               ROUND(AVG(dam_price_usd_mwh), 2) AS average_price_usd_mwh
        FROM market_features
        GROUP BY year
        ORDER BY year
    ''', connection)
annual

## 2. Price-spike conditions

A spike means an hourly price in the full sample's top 5%. `Extreme load` is the top load decile within its own year and `Low renewable` is the bottom renewable-output tercile within that year. These are associations, not causal estimates.

In [ ]:
(market.groupby(['load_regime', 'renewable_regime'], as_index=False)
       .agg(observations=('price_spike_p95', 'size'),
            median_price_usd_mwh=('dam_price_usd_mwh', 'median'),
            spike_rate=('price_spike_p95', 'mean'))
       .sort_values(['load_regime', 'renewable_regime']))

## 3. Reproducible figures

Run `python src/analysis.py` from the project root to regenerate the six figures in `figures/`.